# Asynchronous API-based LLM prompting

When we use LLMs for annotations or other tasks where we want to collect responses for many inputs, doing so sequentially can be slow when we rely on API-based inference.

The naive strategy of iterating over inputs one at a time and waiting for each response can be very slow.

With **asynchronous API calls**, we can send multiple requests concurrently.
This means that instead of waiting for each request individually, up to `batch_size` requests are processed at the same time in parallel.

The returned list follows the **input order**, even if requests finish in a different order.


## Setup

## Client setup

As in the housing exercise, load `HF_TOKEN` from the project's `.env` file.
This uses the OpenAI Python client to call **Hugging Face**, with the same model and provider.

In [1]:
import os
hf_token = os.environ["HF_TOKEN"]

In [2]:
import asyncio
from openai import AsyncOpenAI # NOTE: we import the asynchronous client for OpenAI API calls

openai_client = AsyncOpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=hf_token,
    timeout=60,
    max_retries=0,  # Make request failures visible in this minimal example.
)

In [3]:
MODEL_ID = "Qwen/Qwen2.5-72B-Instruct"
PROVIDER = "deepinfra"
model_id = f"{MODEL_ID}:{PROVIDER}"

## Example Usage

We re-use the task instructions for social media post sentiment analysis we already used in [llm_inference_basics_api.ipynb](./llm_inference_basics_api.ipynb):

In [4]:
task_instruction = (
    "You classify the sentiment expressed in politicians' social media posts. "
    "Identify the sentiment as positive, negative, or neutral. "
    "Only return the classification result without any explanation."
)

## One asynchronous request

We define a functions for making a single asynchronous request to the API.
Here, the function excecutes the LLM inference logic for a single text given the task instructions and the client defined above.

`async def` defines a "coroutine."
A **coroutine** is a special type of function that can pause and resume its execution, allowing asynchronous operations.

In [5]:
async def classify_text(text, task_instruction):
    messages = [
        {"role": "system", "content": task_instruction},
        {"role": "user", "content": f'"""{text}"""'},
    ]
    response = await openai_client.chat.completions.create(
        model=model_id,
        messages=messages,
        # NOTE: hard-coded generation args (could be passed as **kwargs instead)
        max_tokens=1,
        temperature=0,
    )
    content = response.choices[0].message.content
    return content.strip() if content is not None else None

As shown below, we can call the `classify_text` function with `await` to perform asynchronous inference.

## Process a list in batches

`asyncio.gather` (here `tqdm_asyncio.gather`) runs the requests in a batch concurrently and returns their results in input order.
`extend` appends those results to the overall list.

In [6]:
from tqdm.auto import tqdm
from tqdm.asyncio import tqdm_asyncio

async def classify_texts(texts, task_instruction, batch_size=5):
    if not isinstance(batch_size, int) or isinstance(batch_size, bool) or batch_size < 1:
        raise ValueError("batch_size must be a positive integer.")

    classifications = []
    starts = range(0, len(texts), batch_size)

    for batch_number, start in enumerate(
        tqdm(starts, desc="Batches", position=0),
        start=1,
    ):
        batch = texts[start:start + batch_size]

        batch_results = await tqdm_asyncio.gather(
            *(classify_text(text, task_instruction) for text in batch),
            desc=f"Batch {batch_number}",
            position=1,
            leave=False,
        )
        classifications.extend(batch_results)

    return classifications

## Use it with a list of example texts

Reuse your completed `task_instruction` and the `texts` data frame from the housing
exercise in the same notebook session, or copy the functions above into that notebook.

In [7]:
# TODO: load a real long list of texts from your data source
texts = [
    "Great news for our town! Today the council approved funding for a new public library. I am delighted that we can give everyone more places to learn, meet, and connect. Proud of what we have achieved together!",
    "I am deeply concerned about the recent policy changes. They will negatively impact our community and hinder progress.",
    "The new transportation plan seems adequate. It has both positive and negative aspects, and its overall effect remains to be seen.",
    "I am excited about the upcoming community events. They provide a great opportunity for residents to engage and participate actively in local initiatives.",
    "I am worried about the increasing pollution levels in our city. Immediate action is required to protect the environment and public health.",
    "I am thrilled about the recent improvements in our local parks. They offer more recreational spaces for families and contribute to the overall well-being of our community.",
    "I am concerned about the lack of affordable housing in our city. It is crucial to address this issue to ensure that all residents have access to safe and affordable living conditions.",
]


classifications = await classify_texts(
    texts,
    task_instruction,
    batch_size=3,
)

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

The function above processes texts in batches.
That means that texts are grouped into batches of `batch_size` texts.
Within each batch, the texts are processed in parallel because the LLM requests are run concurrently.

::: {.callout-warning title="Bottlenecks"}

Batch-wise processing still means that a single slow request within a batch can become a bottleneck, delaying the processing of the entire batch.

:::

::: {.callout-tip title="Use await directly in a notebook"}

Jupyter already runs an event loop. Use the top-level `await` above.
There is no need for `asyncio.run()` or `nest_asyncio` inside the notebook.
Apply the same label checks as in the sequential exercise.

:::

::: {.callout-warning title="Concurrency is not a rate limit"}

`batch_size` limits simultaneous requests, not requests per minute. Reduce it if the
provider rejects requests. This minimal version raises API errors rather than replacing
failures with labels. Other requests in the current batch may still finish after an error,
but later batches will not start.

:::

## Using a _Semaphore_

A semaphore makes sure that no more than `batch_size` requests run at the same time.
As soon as one request finishes, the next waiting request can start. It does not have
to wait for all the other active requests to finish.

Here, `batch_size` means the **maximum number of requests running at once**. It does
not divide the texts into fixed batches.

In [8]:
async def classify_texts_concurrently(texts, task_instruction, batch_size=5):
    if not isinstance(batch_size, int) or isinstance(batch_size, bool) or batch_size < 1:
        raise ValueError("batch_size must be a positive integer.")

    semaphore = asyncio.Semaphore(batch_size)

    async def classify_with_slot(text):
        # Wait for a free slot. Release it automatically, including on errors.
        async with semaphore:
            return await classify_text(text, task_instruction)

    return await tqdm_asyncio.gather(
        *(classify_with_slot(text) for text in texts),
        desc="Classifying texts",
    )

The progress bar counts completed requests. The returned labels still follow the
input order, regardless of completion order.

In [9]:
classifications = await classify_texts_concurrently(
    texts,
    task_instruction,
    batch_size=3,
)
classifications

Classifying texts: 100%|██████████| 7/7 [00:43<00:00,  6.26s/it]


['positive',
 'negative',
 'neutral',
 'positive',
 'negative',
 'positive',
 'negative']

::: {.callout-warning title="Concurrency and errors"}

The semaphore limits active requests, not requests per minute.
If a request raises an error, this function raises it too!

Other scheduled requests may continue running, so avoid immediately rerunning the cell and submitting the same texts again.

:::

## Retry temporary errors

Some errors arising during API requests are temporary. For example, the provider may be busy, receive too many requests at once, or briefly lose the connection.

The `tenacity` package can help us wait and try these requests again.
To use it, we first need to define the list of "transient" (i.e., temporary) exceptions that should trigger a retry.

```{bash eval=FALSE}
pip install tenacity==9.1.4
```

In [10]:
from openai import (
    APIConnectionError,
    APITimeoutError,
    InternalServerError,
    RateLimitError,
)
TRANSIENT_EXCEPTIONS = (
    RateLimitError,
    APIConnectionError,
    APITimeoutError,
    InternalServerError,
)

The function below then uses `tenacity` to retry transient errors and makes at most six attempts for each text.
The delay grows after each failed attempt and includes a small random variation. Errors that are unlikely to disappear by waiting, such as an invalid API key, are not retried.

The semaphore is inside the retrying function.
This means that a request gives up its slot while it waits before trying again, allowing another request to use that slot.

_Note:_ we still use a semaphore to limit the number of concurrent requests, even though requests may release their slot temporarily while waiting to retry.

In [11]:
from tenacity import (
    retry,
    retry_if_exception_type,
    stop_after_attempt,
    wait_exponential_jitter,
)

async def classify_texts_patiently(texts, task_instruction, batch_size=5):
    if not isinstance(batch_size, int) or isinstance(batch_size, bool) or batch_size < 1:
        raise ValueError("batch_size must be a positive integer.")

    semaphore = asyncio.Semaphore(batch_size)

    @retry(
        retry=retry_if_exception_type(TRANSIENT_EXCEPTIONS),
        wait=wait_exponential_jitter(initial=1, max=60),
        stop=stop_after_attempt(6),
        reraise=True,
    )
    async def classify_with_slot_patiently(text):
        async with semaphore:
            return await classify_text(text, task_instruction)

    return await tqdm_asyncio.gather(
        *(classify_with_slot_patiently(text) for text in texts),
        desc="Classifying texts patiently",
    )

Run the classification over the texts again, this time with automatic retries:

In [12]:
classifications = await classify_texts_patiently(
    texts,
    task_instruction,
    batch_size=3,
)

Classifying texts patiently: 100%|██████████| 7/7 [01:03<00:00,  9.05s/it]


::: callout-warning

If a temporary error continues through all six attempts, the function still raises
the error.
Retries make brief problems easier to handle, but they do not enforce a requests-per-minute limit.

:::

## Clean up

Close the client when finished. Rerun the setup cell if you want to use it again.

In [13]:
await openai_client.close()